# Training a Profile Predictor


## Model Overview
Given a set of 0-D scalar inputs, we want to predict the $T_e$ and $n_e$ profiles.

Let's take a look at the input and output structures. You'll notice some properties that do simple calculations on the input structure; this is mostly for approximation and normalization purposes to try and make NN training work better.

In [ ]:
%load_ext autoreload
%autoreload 2

import inspect

from IPython.display import Markdown, display

from popsim.modules.profile_predictor.module import Inputs, Outputs, ProfilePredictor

display(Markdown(f"```python\n{inspect.getsource(Inputs)}\n```"))
display(Markdown(f"```python\n{inspect.getsource(Outputs)}\n```"))

Okay great that's the input and output structures. Now let's take a look at the model architecture along with the currently implemented options.

Below, we can see that there is a `coeff_predictor` that maps the 0D inputs to coefficients for profile modes. Currently, we have two different kinds of profile mode representations: a PCA-like approach, and a convex combination. In both cases, the profile shape is represented as such:


$$\frac{n_e(\rho)}{\bar{n}_e(\rho)} = \sum_{i}^{n_{shapes}} c_{n,i} n_{shape, i}(\rho) \quad \frac{T_e(\rho)}{\bar{T}_e(\rho)} = \sum_{i}^{n_{shapes}} c_{T,i} T_{shape, i}(\rho)$$

In the PCA case, the coefficients are un-constrained. In the convex combination case, a `softmax` function is applied to guarantee that the weights sum to 1. This has the appeal of enabling an interpertation of the weights as specifying which "mode" we are currently in. It also allows for us to normalize $n_{shape, i}$ to have an integral of 1, which also provides some guarantees on the model behavior.

The line averaged density is an input, so the shapes can be scaled by it. The electron temperature profile is determined by an ad-hoc approximate formula to get us within the right order of magnitude multiplied by a NN correction.

In [ ]:
display(Markdown(f"```python\n{inspect.getsource(ProfilePredictor.__call__)}\n```"))

## Defining our Training Configuration

Here is the configuration dictionary that will configure our training runs.

In [ ]:
from pprint import pformat

from IPython.display import Markdown, display

from popsim.modules.profile_predictor.train_configs import SPARC_CONFIG

# Format the dictionary with pprint
formatted_config = pformat(SPARC_CONFIG, indent=4, width=80)
display(Markdown(f"```python\n{formatted_config}\n```"))

## Defining a Module Evaluation Environment
As usual, we're going to wrap our module to define how data flows in.

Something you'll notice is the presence of `get_trainable` method with the `freeze_shapes` option to freeze the shapes (i.e. train the rest of the model, but do not train the shapes). This is because `ProfilePredictor.init` does some stuff to figure out what are good initial guesses for shapes (using PCA for `ShapeType.PCA_LIKE` and K-Means clustering for `ShapeType.CONVEX_COMBINATION`). This gives you the option to choose whether or not you want the optimization process to learn both the shapes and the mapping of the 0D scalars to shapes.

In [ ]:
from popsim.modules.profile_predictor.module import EvalEnv

display(Markdown(f"```python\n{inspect.getsource(EvalEnv)}\n```"))

## Defining Evaluation Suites

Currently, an "evaluation function" is defined as a function that takes in a `EvalData` datastructure and outputs something (dictionary, figure, what have you) that reports on the performance of a model.

An evaluation suite, is then just a dictionary of evaluation functions.

Below, we define a couple of evaluation functions and two suites:
1. One to run regularly during training
2. One to run at the end of training for testing

In [ ]:
from popsim.modules.profile_predictor import evals

test_eval_suite = {
    "violin_shapes_in_data": evals.violin_shapes_in_data,
    "integrated_profile_error": evals.compute_integrated_error,
    "compare_profiles": evals.compare_profiles_for_episode,
    "make_histograms": evals.make_histograms,
    "plot_shapes": evals.plot_shapes,
    "make_quantile_examples": evals.make_quantile_examples,
    "plot_ne_te_weights": evals.plot_ne_te_weights,
}

## Defining the Train Function and Training a Model

In [ ]:
from popsim.modules.profile_predictor.train import train

trainer, train_dl, val_dl, test_dl = train(SPARC_CONFIG)

## Running the Evaluation Suite and Getting Results

In [ ]:
trainer.run_evals(test_dl, test_eval_suite)

## Getting the Input and Output datasets for the Model
We can just call run_evals without an evaluation suite to the `EvalData` structure which will give us the input and output datasets.

In [ ]:
eval_data = trainer.run_evals(test_dl)
eval_data.output_ds

## Exporting the Model as a JSON
The export functionality will save the model as a JSON and also create a netcdf file that contains input + output data for validation.

In [ ]:
import os

from popsim import PACKAGE_ROOT
from popsim.ml import export

path_export_dir = os.path.join(PACKAGE_ROOT, "modules", "profile_predictor", "exports", "sparc_latest")

export(trainer.train_state.model, path_export_dir, test_dl)